
# Using GEE and satellite embeddings for predictive mapping
Author: Antoine Stevens, Kristof Van Oost

We will learn today how to do supervised classification workflow to map mangrove forests along the Kenyan coastline using **Google Earth Engine**. Source of this demo is [here](https://developers.google.com/earth-engine/tutorials/community/satellite-embedding-03-supervised-classification)

The primary goal of this session is to show how to leverage the [Google Satellite Embedding dataset](https://deepmind.google/blog/alphaearth-foundations-helps-map-our-planet-in-unprecedented-detail/) to produce a decent-quality mangrove classification map for the year 2020 with minimal training data.

### What are spatial/satellite embeddings ?

* In natural language processing, **word embeddings** represent words as vectors in a mathematical space

  * This works through an encoding process learned from large text corpora such as Wikipedia or news articles. An encoder (e.g., a neural network) takes a word and maps it into a fixed-size vector of for example 100 or 1000 dimensions.
  * During training, the model adjusts these **vectors** so that words appearing in similar contexts end up with similar representations. For example, the model would learn that “cat” and “dog” appear often in similar contexts, so their vectors become close. 
  * In modern models (like transformers), embeddings are often **contextual**: the same word can map to different vectors depending on the sentence. This is achieved by passing token embeddings through multiple layers of neural networks (attention layers), which refine the representation based on surrounding words.

* **Spatial embeddings** extend the same idea to geographic data. Instead of representing words, they represent places, image patches, or other spatial entities as vectors. The goal is to encode meaningful properties of a location such as land cover, environmental context, texture, seasonality, etc. in a compact numerical form.

* A specific case is **satellite embeddings**: vectors derived from satellite imagery by a machine-learning model, usually a deep neural network trained on large volumes of Earth observation data. In that sense, they can be seen as a kind of foundation model for Earth observation: just as language models compress patterns in text, satellite embedding models compress patterns in imagery and geospatial signals.

  * Instead of working directly with raw spectral bands such as Sentinel-2 values, one works with **precomputed features** that already capture patterns such as vegetation structure, water-land boundaries, urban texture, radar response, or seasonal dynamics. This makes the data easier to compare, classify, and analyze, at the cost of some interpretability compared with raw bands.

![Embeddings.png](images/embeddings.png) 

A global embedding field broken down into a single embedding ([Source](https://deepmind.google/blog/alphaearth-foundations-helps-map-our-planet-in-unprecedented-detail/))

  * One example is [AlphaEarth](https://arxiv.org/pdf/2507.22291), where each 10 m × 10 m pixel is encoded as a **64-dimensional vector** summarizing the signal extracted from multiple Earth observation sources (Table 1). A core strength is that it is **multimodal**: it is trained not only on optical imagery, but also on radar, elevation, LiDAR, climate variables, land-cover products, and geocoded text
  * Once locations are represented as vectors, a wide range of machine-learning methods become possible:

    * **Similarity search**: given one location, we can retrieve other locations with nearby embeddings and therefore similar characteristics. 
    * **Supervised learning**: embeddings can be used as input features for classifiers or regressors to predict land cover, crop type, biomass, flood risk, or other variables with relatively little labeled data. 
    * **Unsupervised learning**: embeddings can be clustered to reveal recurring landscape types or hidden spatial patterns

Table 1. AlphaEarth training data sources ([source](https://arxiv.org/pdf/2507.22291))

| Type | Dataset | Product | Bands / Features | Resolution (m) | Usage |
|---|---|---|---|---:|---|
| Optical | Sentinel-2 | L1C | B2 (Blue), B3 (Green), B4 (Red), B8 (NIR), B11 (SWIR) | 10, 20, 60 | input, target |
| Optical, Thermal | Landsat-8 / Landsat-9 | L1C | B2 (Blue), B3 (Green), B4 (Red), B5 (NIR), B6 (SWIR), B8 (Panchromatic), B10 (Thermal) | 15, 30, 100 | input, target |
| C-band SAR | Sentinel-1A / Sentinel-1B | GRD | VV, VH, HH, HV, angle | 10 | input, target |
| L-band SAR | ALOS PALSAR ScanSAR | Level 2.2 | HH, HV, lin | 25 | target |
| Elevation | Copernicus DEM | GLO-30 | DEM (elevation) | 30 | target |
| LiDAR | GEDI | L2A | Relative height metrics (rh*) | 25 | target |
| Climate | ERA5-Land | Monthly aggregates | Total precipitation (sum, min, max); air temperature at 2 m (mean, min, max); dewpoint temperature at 2 m (mean, min, max); surface pressure (mean, min, max) | 11132 | target |
| Gravity fields | GRACE | Monthly mass grids | Equivalent liquid water thickness | 11132 | target (@50%) |
| Land cover | National Land Cover Database | NLCD 2019, 2021 | Landcover | 30 | target (@50%) |
| Text | Wikipedia geocoded articles | — | Text embeddings | N/A | target |
| Text | GBIF research-grade observations | — | Text embeddings (class, genus, species) | N/A | target |

Note: We will use [AlphaEarth model](https://arxiv.org/pdf/2507.22291) today as it is directly available in [GEE](https://developers.google.com/earth-engine/datasets/catalog/GOOGLE_SATELLITE_EMBEDDING_V1_ANNUAL), but there are other similar projects, including:
* [TerraMind](https://www.esa.int/Applications/Observing_the_Earth/ESA_and_IBM_collaborate_on_TerraMind)  
* [Tessera](https://arxiv.org/html/2506.20380v4)
* [Prithvi-EO](https://research.ibm.com/publications/prithvi-eo-an-open-access-geospatial-foundation-model-advancing-earth-science-through-global-collaboration) 


## Methodology

In this demo we will follow 5 steps:

1.  **Define an Area of Interest (AOI):** A region along the coast of Kenya.
2.  **Collect Training Data:** A small number of points are defined for 'Mangrove', 'Water', and 'Other' land cover classes.
3.  **Train a Classifier:** A k-Nearest Neighbors (kNN) classifier is trained on the satellite embedding vectors using the collected training points.
4.  **Classify:** The entire AOI is classified using the trained model.
5.  **Validate:** The resulting mangrove map is compared against the Global Mangrove Watch (GMW) dataset for the same year to assess accuracy.

In [ ]:
# Load modules and authenticate
import ee
import geemap

# Initialize Earth Engine
ee.Authenticate(auth_mode="localhost")
ee.Initialize(project="proj-01kcrxt2hcj8a")

## 1. Define Study Area

In [2]:
# Define region of interest - Kenyan Coastline
geometry = ee.Geometry.Polygon([[
  [39.4926, -4.39833],
  [39.4926, -4.47394],
  [39.5491, -4.47394],
  [39.5491, -4.39833]
]])

# Create base map
Map = geemap.Map()
Map.centerObject(geometry, zoom=13)
Map.addLayer(geometry, {'color': 'FF0000'}, 'Study Area (Kenya)')
Map

Map(center=[-4.436134891994578, 39.520849999999875], controls=(WidgetControl(options=['position', 'transparent…

## 2. Collect Training Samples

For this demo, we'll create sample points programmatically for three classes: - **Mangroves** (class 1) - **Water** (class 2) - **Other** (class 3)

In [3]:
# Define a year for the analysis
year = 2020
start_date = ee.Date.fromYMD(year, 1, 1)
end_date = start_date.advance(1, 'year')

# Load satellite embeddings for the specified year
embeddings = ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')
embeddings_filtered = embeddings \
    .filter(ee.Filter.date(start_date, end_date)) \
    .filter(ee.Filter.bounds(geometry))
embeddings_image = embeddings_filtered.mosaic()

# Create training samples for the new AOI in Kenya
# Mangrove samples (coastal vegetation)
mangroves = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([39.53332459156346, -4.424306862786138]), {'landcover': 1}),
    ee.Feature(ee.Geometry.Point([39.515674132076356, -4.417484208564611]), {'landcover': 1}),
    ee.Feature(ee.Geometry.Point([39.521682280269715, -4.414916932798933]), {'landcover': 1}),
    ee.Feature(ee.Geometry.Point([39.53301193114862, -4.412092919198809]), {'landcover': 1}),
    ee.Feature(ee.Geometry.Point([39.52305557128534, -4.41904596364173]), {'landcover': 1}),
    ee.Feature(ee.Geometry.Point([39.52751876708612, -4.418767843115292]), {'landcover': 1}),
    ee.Feature(ee.Geometry.Point([39.512952520329506, -4.42470302325086]), {'landcover': 1}),
    ee.Feature(ee.Geometry.Point([39.51191450544074, -4.425050671107723]), {'landcover': 1}),
    ee.Feature(ee.Geometry.Point([39.51075895498842, -4.429791895347292]), {'landcover': 1}),
    ee.Feature(ee.Geometry.Point([39.51114519308656, -4.429235662261579]), {'landcover': 1}),
])

# Water samples
water = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([39.51274523032135, -4.4343394780686385]), {'landcover': 2}),
    ee.Feature(ee.Geometry.Point([39.51441892874664, -4.429162247319931]), {'landcover': 2}),
    ee.Feature(ee.Geometry.Point([39.52403196585602, -4.429418970360026]), {'landcover': 2}),
    ee.Feature(ee.Geometry.Point([39.52016958487457, -4.425782052317534]), {'landcover': 2}),
    ee.Feature(ee.Geometry.Point([39.50879701865143, -4.443581386549747]), {'landcover': 2}),
    ee.Feature(ee.Geometry.Point([39.50553545248932, -4.447860008667267]), {'landcover': 2}),
    ee.Feature(ee.Geometry.Point([39.50399050009674, -4.449913738457232]), {'landcover': 2}),
    ee.Feature(ee.Geometry.Point([39.51965460074371, -4.445635128266346]), {'landcover': 2}),
    ee.Feature(ee.Geometry.Point([39.51171526205963, -4.469081606555194]), {'landcover': 2}),
    ee.Feature(ee.Geometry.Point([39.50347551596588, -4.469680594368735]), {'landcover': 2}),
    ee.Feature(ee.Geometry.Point([39.525425885851405, -4.417474500749167]), {'landcover': 2}),
])

# Other land cover samples (bare land, other vegetation)
other = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([39.50560992076606, -4.4242595999510455]), {'landcover': 3}),
    ee.Feature(ee.Geometry.Point([39.50620000674934, -4.423307578260568]), {'landcover': 3}),
    ee.Feature(ee.Geometry.Point([39.50448339297981, -4.423799634568]), {'landcover': 3}),
    ee.Feature(ee.Geometry.Point([39.50197284534187, -4.423649878335117]), {'landcover': 3}),
    ee.Feature(ee.Geometry.Point([39.503313949849314, -4.4228369153997535]), {'landcover': 3}),
    ee.Feature(ee.Geometry.Point([39.497137934254695, -4.416632173880207]), {'landcover': 3}),
    ee.Feature(ee.Geometry.Point([39.49610796599298, -4.418386475891598]), {'landcover': 3}),
    ee.Feature(ee.Geometry.Point([39.504884153889705, -4.415434111585769]), {'landcover': 3}),
    ee.Feature(ee.Geometry.Point([39.540219677644, -4.402738757778825]), {'landcover': 3}),
    ee.Feature(ee.Geometry.Point([39.537083376051925, -4.414620272781714]), {'landcover': 3}),
    ee.Feature(ee.Geometry.Point([39.54329128212492, -4.400688640340887]), {'landcover': 3}),
    ee.Feature(ee.Geometry.Point([39.543870639272136, -4.399725891211968]), {'landcover': 3}),
    ee.Feature(ee.Geometry.Point([39.50106019752159, -4.41120573786903]), {'landcover': 3}),
    ee.Feature(ee.Geometry.Point([39.50085098521843, -4.411943834251456]), {'landcover': 3}),
])

# Merge all training samples
gcps = mangroves.merge(water).merge(other)

# Create a map to visualize the embeddings and training points
Map2 = geemap.Map()
Map2.centerObject(geometry, zoom=13)

# Create a false-color composite of the embeddings for visualization
embedding_vis = {'bands': ['A01', 'A16', 'A09'], 'min': -0.5, 'max': 0.5}
Map2.addLayer(embeddings_image.clip(geometry), embedding_vis, 'Satellite Embeddings (False Color)')

# Add styled training points - style each class separately
mangrove_style = mangroves.style(color='00FF00', pointSize=5, fillColor='00FF00')
water_style = water.style(color='0000FF', pointSize=5, fillColor='0000FF')
other_style = other.style(color="#282828", pointSize=5, fillColor='282828')

Map2.addLayer(mangrove_style, {}, 'Training Points - Mangroves')
Map2.addLayer(water_style, {}, 'Training Points - Water')
Map2.addLayer(other_style, {}, 'Training Points - Other')

# Add a legend
legend_dict = {'Mangroves': '00FF00', 'Water': '0000FF', 'Other': '282828'}
Map2.add_legend(legend_title="Training Classes", legend_dict=legend_dict)

print(f"Total training samples: {gcps.size().getInfo()}")
Map2

Total training samples: 35


Map(center=[-4.436134891994578, 39.520849999999875], controls=(WidgetControl(options=['position', 'transparent…

## 3. Train Classifier Using Satellite Embeddings

In [4]:
# Sample the embeddings at training points
training = embeddings_image.sampleRegions(
    collection=gcps,
    properties=['landcover'],
    scale=10
)

# Train a K-Nearest Neighbors classifier
classifier = ee.Classifier.smileKNN().train(
    features=training,
    classProperty='landcover',
    inputProperties=embeddings_image.bandNames()
)

print("Classifier trained successfully!")

Classifier trained successfully!


## 4. Classify and Visualize Results

In [5]:
# Classify the satellite embeddings
classified = embeddings_image.classify(classifier)

# Visualize the classification
# Palette: Mangrove (green), Water (blue), Other (gray)
palette = ['00FF00', '0000FF', '808080']

Map3 = geemap.Map()
Map3.centerObject(geometry, zoom=13)
Map3.addLayer(
    classified.clip(geometry),
    {'min': 1, 'max': 3, 'palette': palette},
    'Classified Land Cover'
)

# Extract and display mangroves class only
mangroves_image = classified.eq(1).selfMask()
mangrove_vis = {'min': 0, 'max': 1, 'palette': ['00FF00']}

Map3.addLayer(
    mangroves_image.clip(geometry),
    mangrove_vis,
    'Mangroves (Satellite Embedding Classification)'
)

# Add a legend
legend_dict = {
    'Mangroves': '00FF00',
    'Water': '0000FF',
    'Other': '808080'
}
Map3.add_legend(legend_title="Land Cover Classes", legend_dict=legend_dict)

Map3

Map(center=[-4.436134891994578, 39.520849999999875], controls=(WidgetControl(options=['position', 'transparent…

## 5. Compare with Global Mangrove Watch

In [6]:
# Load Global Mangrove Watch data for comparison
gmw_image = ee.Image("projects/sat-io/open-datasets/GMW/extent/GMW_V3/gmw_v3_2020")

# Prepare binary mangrove layers for comparison
# Use unmask(0) to make sure the images have values everywhere for calculation
our_mangroves = mangroves_image.unmask(0)
gmw_mangroves = gmw_image.gt(0).unmask(0)

# Create a difference image
# 1 = Agreement (Both our classification and GMW show mangroves)
# 2 = Our Classification Only (Commission error)
# 3 = GMW Only (Omission error)
diff_image = ee.Image(0).clip(geometry)
diff_image = diff_image.where(our_mangroves.eq(1).And(gmw_mangroves.eq(1)), 1)
diff_image = diff_image.where(our_mangroves.eq(1).And(gmw_mangroves.eq(0)), 2)
diff_image = diff_image.where(our_mangroves.eq(0).And(gmw_mangroves.eq(1)), 3)
diff_image = diff_image.selfMask()

# Create a new map for toggleable layers
Map4 = geemap.Map()
Map4.centerObject(geometry, zoom=13)

# Satellite basemap underneath
Map4.add_basemap("SATELLITE")

# Add our classification layer (toggleable)
Map4.addLayer(
    mangroves_image.clip(geometry),
    mangrove_vis,
    'Our Classification',
    shown=False
)

# Add GMW reference data layer (toggleable)
gmw_vis = {'min': 0, 'max': 1, 'palette': ["#E400E4"]}
Map4.addLayer(
    gmw_image.clip(geometry).gt(0).selfMask(),
    gmw_vis,
    'Global Mangrove Watch',
    shown=False  # Initially hidden
)

# Add the difference layer (toggleable)
diff_palette = ['#008000', '#FFA500', '#FF0000'] # Green, Orange, Red
diff_vis = {'min': 1, 'max': 3, 'palette': diff_palette}
Map4.addLayer(
    diff_image,
    diff_vis,
    'Difference Layer',
    shown=True
)

# Add a legend for the difference layer
diff_legend_dict = {
    'Agreement (Both)': '008000',
    'Our Classification Only': 'FFA500',
    'GMW Only': 'FF0000'
}
Map4.add_legend(legend_title="Comparison Legend", legend_dict=diff_legend_dict, position='bottomright')

Map4

Map(center=[-4.436134891994578, 39.520849999999875], controls=(WidgetControl(options=['position', 'transparent…

## 6. Calculate Area Statistics

In [7]:
# Calculate mangrove area from our classification
def calculate_area_ha(image, geometry):
    area = image.multiply(ee.Image.pixelArea()).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=geometry,
        scale=10,
        maxPixels=1e10
    )
    return ee.Number(area.values().get(0)).divide(10000)

# Calculate areas
classified_area = calculate_area_ha(mangroves_image, geometry)
gmw_area = calculate_area_ha(gmw_image.gt(0), geometry)

diff = classified_area.subtract(gmw_area).abs()
pct = diff.divide(gmw_area).multiply(100)

# Pull results ONCE
vals = ee.Dictionary({
    "classified": classified_area,
    "gmw": gmw_area,
    "diff": diff,
    "pct": pct
}).getInfo()

print("Mangrove Area Results for Kenyan AOI:")
print("------------------------------------")
print(f"Our Classification: {vals['classified']:.2f} hectares")
print(f"Global Mangrove Watch: {vals['gmw']:.2f} hectares")
print(f"Difference: {vals['diff']:.2f} hectares | {vals['pct']:.1f}%")

Mangrove Area Results for Kenyan AOI:
------------------------------------
Our Classification: 666.42 hectares
Global Mangrove Watch: 682.72 hectares
Difference: 16.30 hectares | 2.4%


We can see that the results are already quite strong, even with only a **small number of labeled examples**!

This illustrates a form of **few-shot learning**. A key advantage of embeddings is that the difficult work of feature extraction has already been done by the pretrained foundation model. As a result, downstream tasks often require much less labeled data.

Instead of training directly on raw imagery, we can train a lightweight classifier on top of the embeddings using only a small set of examples since the **representation is already structured in a meaningful way**, so the downstream model mainly has to learn the decision boundary.

